# Enterprise 08 — Agent Security Architecture Review Workshop

**Goal:** transform an incomplete design packet into reproducible findings,
verify the repairs, and preserve honest review metrics.

![Architecture review evidence loop](architecture.svg)

This notebook is deterministic, credential-free, and makes no external calls.
Its local HMAC proves fixture integrity only; it is not deployment authority.

## Security boundary

Humans and models may propose threats. Trusted code parses the packet, derives
scope, evaluates invariants, admits evidence, records `PASS`, `BLOCK`, or
`UNKNOWN`, and verifies a new version. Schema validity and reviewer confidence
do not establish trust.

In [ ]:
from dataclasses import asdict
import json

import architecture_review as review
import review_fixtures as fixtures
from review_schema_adapter import parse_packet
import lab

packet = fixtures.flawed_packet()
print({"packet": packet.packet_id, "version": packet.version, "flows": len(packet.flows)})

## 1. Parse the untrusted packet

Pydantic v2 rejects unknown fields and coercion. The result still needs tenant,
provenance, freshness, and invariant checks.

In [ ]:
parsed = parse_packet(asdict(packet))
assert parsed == packet
print({"assets": len(parsed.assets), "evidence": len(parsed.evidence)})

## 2. Run the diagram-only baseline

All endpoints exist, so a topology-only review reports no findings. It never
asks whether authority, tenant, effects, failure, evidence, or ownership are safe.

In [ ]:
naive = review.diagram_only_review(packet)
print({"diagram_only_findings": naive})
assert naive == ()

## 3. Trace paths against deterministic invariants

The structured review evaluates every authority-changing and effect path and
binds its disposition to the exact packet digest and rule version.

In [ ]:
service = fixtures.service()
result = service.review(packet, fixtures.NOW)
print(json.dumps([asdict(item) for item in result.findings], indent=2, default=lambda value: value.value))
assert result.receipt.disposition is review.Disposition.BLOCK
assert {item.key for item in result.findings} == fixtures.expected_keys()

## 4. Inspect an indirect encoded effect

The capability text is encoded, but the declared target and effect semantics
still expose missing effect controls, fail-open behavior, stale evidence, and
missing ownership. Keyword filtering would miss this path.

In [ ]:
encoded = [item for item in result.findings if item.flow_id == "encoded-export"]
print([item.invariant_id for item in encoded])
assert len(encoded) == 4

## 5. Verify the review receipt

The receipt detects fixture tampering and binds the decision to packet version,
tenant, rules, time, and finding keys. It is evidence—not approval or authority.

In [ ]:
print(asdict(result.receipt))
assert service.verify_receipt(result.receipt)
assert result.receipt.packet_digest == review.packet_digest(packet)

## 6. Preserve uncertainty during evidence outage

Unavailable evidence cannot become an empty-finding pass. The explicit
`UNKNOWN` state routes the packet to reconciliation and a later re-review.

In [ ]:
unknown = fixtures.service(evidence_available=False).review(packet, fixtures.NOW)
print(asdict(unknown.receipt))
assert unknown.receipt.disposition is review.Disposition.UNKNOWN

## 7. Verify a versioned remediation

The repaired packet advances from version 3 to 4, corrects every path, and
attaches current producer-authenticated control tests. Ticket status alone
would not prove closure.

In [ ]:
fixed = fixtures.remediated_packet()
final = service.review(fixed, fixtures.NOW)
outcomes = review.verify_remediations(service, packet, fixed, result.findings, fixtures.NOW)
print(json.dumps([asdict(item) for item in outcomes], indent=2))
assert final.receipt.disposition is review.Disposition.PASS
assert all(item.verified for item in outcomes)

## 8. Evaluate distinct populations

Precision, severe-gap recall, remediation quality, and reviewer agreement answer
different questions. Every value retains its numerator and denominator; a
zero-size population is `None`.

In [ ]:
evaluation = lab.evaluate_workshop()
print(json.dumps(asdict(evaluation), indent=2))
assert evaluation.finding_precision.value == 1.0
assert evaluation.severe_gap_recall.value == 1.0
assert evaluation.naive_severe_gap_recall.value == 0.0
assert evaluation.remediation_quality.value == 1.0
assert evaluation.reviewer_agreement.value == 0.8

## Checkpoint

Two reviewers agree that mTLS protects a refund flow, and the diagram labels the
service “authorized.” The refund service receives a shared workload identity
and trusts tenant/capability fields supplied by the agent. Is this actionable
review evidence?

**Expected reasoning:** No. Agreement and diagram labels do not prove delegated
authority, resource ownership, exact capability, current policy, approval,
idempotency, or the resulting effect. The finding should name the violated
invariant and attack path; the refund owner must enforce those controls and
produce current negative and postcondition tests before independent re-review.

## Production upgrades and exercises

1. Add a memory-write flow with provenance, retention, deletion, and
   system-of-record precedence.
2. Add a bounded, expiring risk-exception receipt without changing the finding
   to “fixed.”
3. Export stable IDs to Threat Dragon, pytm, or Threagile and reconcile their
   proposed threats with the deterministic finding register.
4. Replace local evidence with an authenticated store and document key rotation,
   revocation, retention, concurrency, and failure behavior.